**Lab Session 2: Building a mini-Blockchain in Python  **

Ahmed Jaidane,  Tahirou laouan magagi

lien gitub : https://github.com/AHKB9/mini-blockchain-python

PART 1 : Cryptographic HASH FUNCTIONS

Question 1–2 : Create a new Python notebook named "mini_blockchain.ipynb"  Import the necessary Python libraries such as hashlib and ecdsa.

A Python notebook is created using Google Colab.
The necessary libraries are imported for hashing and digital signatures.

In [23]:
# Import standard libraries for hashing, data handling, and timestamps
import hashlib
import json
import time

# Install library for digital signatures (only needed in Google Colab)
!pip install ecdsa

# Import ECDSA tools for key generation and signing
from ecdsa import SigningKey, SECP256k1

Question 3 : Implement a Block class to represent a block in the blockchain. The class should include:- block_number - timestamp - previous_hash - transactions - hash. Question 4 : Implement a method calculate_hash() that computes the SHA-256 hash based on the block's attributes.

The Block class represents a block in the blockchain.
It contains attributes such as block number, timestamp, previous hash, and transactions.

The calculate_hash() method computes a SHA-256 hash based on the block data.

In [24]:

class Block:

    # Initialize a block with its main attributes
    def __init__(self, block_number, timestamp, previous_hash, transactions):
        self.block_number = block_number
        self.timestamp = timestamp
        self.previous_hash = previous_hash
        self.transactions = transactions

        # Mining parameters
        self.nonce = 0
        self.difficulty = 2

        # Compute initial hash of the block
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # Prepare block data for hashing
        data = {
            "block_number": self.block_number,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "transactions": self.transactions,
            "nonce": self.nonce
        }

        # Convert data to string and apply SHA-256 hashing
        block_string = json.dumps(data, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

Question 5 Validate the Block class : Create a block and compute its hash. Modify one attribute and recompute the hash. Verify that the hash changes (avalanche effect).

In [25]:
# Create a sample block
block = Block(
    block_number=1,
    timestamp=1773754779,
    previous_hash="0" * 64,
    transactions=["Alice -> Bob : 10 BTC", "Bob -> Charlie : 5 BTC"]
)

original_hash = block.calculate_hash()
print("Original hash :", original_hash)

# Avalanche effect : change the timestamp by 1 second
block.timestamp = 1773754780
modified_hash = block.calculate_hash()
print("Modified hash :", modified_hash)

print("\nHashes are different:", original_hash != modified_hash)

Original hash : f8b7d95d91d7473af921d2378e5ef1092da7516f180f0ef9dfba3a0d679519f1
Modified hash : 6e5b023e54b341a1876d022d34667d995c6459734172a5e0e87283d078982b63

Hashes are different: True


Question 6 : Implement a Blockchain class with the following methods: - add_block() - get_latest_block() - verify_chain()


In [31]:
class Blockchain:

    # Initialize blockchain with a genesis block
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    # Create the first block
    def create_genesis_block(self):
        return Block(0, 0, "0"*64, ["Genesis"])

    # Return the latest block
    def get_latest_block(self):
        return self.chain[-1]

    # Add a new block to the chain
    def add_block(self, block):
        latest = self.get_latest_block()
        block.previous_hash = latest.hash
        block.hash = block.calculate_hash()
        self.chain.append(block)

    # Verify the integrity of the blockchain
    def verify_chain(self):
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.calculate_hash():
                return False

            if current.previous_hash != previous.hash:
                return False

        print("Chain is valid.")
        return True

Question 7 Validate the Blockchain class : Create a blockchain and add multiple blocks. Call get_latest_block() and verify it returns the latest block.
Modify a block's previous_hash and verify that verify_chain() detects the tampering.

In [32]:
bc = Blockchain()

# Add 3 blocks
b1 = Block(1, 1773754779, bc.get_latest_block().hash, ["Alice -> Bob : 10 BTC"])
bc.add_block(b1)

b2 = Block(2, 1773754780, bc.get_latest_block().hash, ["Bob -> Charlie : 5 BTC"])
bc.add_block(b2)

b3 = Block(3, 1773754781, bc.get_latest_block().hash, ["Charlie -> Dave : 2 BTC"])
bc.add_block(b3)

# get_latest_block
latest = bc.get_latest_block()
print(f"\nLatest block: #{latest.block_number} | hash: {latest.hash}")

# verify_chain on intact chain
print("\n--- Verifying intact chain ---")
bc.verify_chain()

# Tamper with block 2 and verify again
print("\n--- Tampering block 2 then verifying ---")
bc.chain[2].previous_hash = "deadbeef" * 8
bc.verify_chain()


Latest block: #3 | hash: 87e968875597a3a5107ef65e2b860e69ebe515d61623d62e7f12617dd5b62b7d

--- Verifying intact chain ---
Chain is valid.

--- Tampering block 2 then verifying ---


False

Question 8 : Implement a method calculate_merkle_root() that computes
the Merkle root of a list of transactions.

In [33]:
def calculate_merkle_root(transactions):
    import hashlib

    hashes = [hashlib.sha256(tx.encode()).hexdigest() for tx in transactions]

    while len(hashes) > 1:
        new_hashes = []

        for i in range(0, len(hashes), 2):
            if i + 1 < len(hashes):
                combined = hashes[i] + hashes[i+1]
            else:
                combined = hashes[i] + hashes[i]

            new_hashes.append(hashlib.sha256(combined.encode()).hexdigest())

        hashes = new_hashes

    return hashes[0]

Question 9 Merkle Root Validate the Merkle Root Calculation :

Create a list of transactions and compute the Merkle root.
Modify one transaction and recompute the root.
Verify that the Merkle root changes.

In [34]:
# Define a list of sample transactions
txs = ["Alice -> Bob", "Bob -> Charlie", "Charlie -> Dave"]

# Compute the original Merkle root
root1 = calculate_merkle_root(txs)
print("Original Merkle Root:", root1)

# Modify one transaction to simulate data tampering
txs[0] = "MODIFIED"

# Recompute the Merkle root after modification
root2 = calculate_merkle_root(txs)
print("Modified Merkle Root:", root2)

# Check if the Merkle roots are different
print("\nMerkle roots are different:", root1 != root2)

Original Merkle Root: 43b569eb89b13632b0361d08ed336c547ccfe082166b6367f2d63c5660948c77
Modified Merkle Root: e0db0d51e21834a2f8637b342c2c02918e7e58d915b81157b46cbb6f6f104329

Merkle roots are different: True


### Part 2 : Digital Signatures


Question 8 : Transaction Class

Implement a Transaction class to represent a transaction in the blockchain.
The class should have the following attributes:

- sender: The address of the sender.
- receiver: The address of the receiver.
- amount: The amount of cryptocurrency being transferred.
- signature: The digital signature of the transaction.

Question 9: Digital Signature Methods

Implement two methods in the Transaction class:

- sign_transaction(private_key): Sign the transaction using ECDSA (Elliptic Curve Digital Signature Algorithm) and the provided private key. Use the ecdsa library to generate the signature and the secp256k1 curve used in Bitcoin and Ethereum (as seen in the course).

- verify_signature(public_key): Verify the transaction's signature using the provided public key. Use the ecdsa library to verify the signature.


In [35]:
# Import required libraries for digital signatures and hashing
from ecdsa import SigningKey, SECP256k1
import hashlib

class Transaction:

    # Initialize a transaction with sender, receiver, amount and empty signature
    def __init__(self, sender, receiver, amount):
        self.sender = sender
        self.receiver = receiver
        self.amount = amount
        self.signature = None

    # Generate a hash of the transaction data using SHA-256
    def calculate_hash(self):
        data = str(self.sender) + str(self.receiver) + str(self.amount)
        return hashlib.sha256(data.encode()).hexdigest()

    # Sign the transaction using the sender's private key
    def sign_transaction(self, private_key):
        tx_hash = self.calculate_hash()
        self.signature = private_key.sign(tx_hash.encode())

    # Verify the transaction signature using the sender's public key
    def verify_signature(self, public_key):
        tx_hash = self.calculate_hash()
        try:
            return public_key.verify(self.signature, tx_hash.encode())
        except:
            return False

Question 10: Wallet Class

Implement a Wallet class to represent a user's wallet.
A wallet is a software application that stores the user's private and public keys,
allowing them to manage their cryptocurrency assets.

The Wallet class should have the following methods:

- generate_keys(): Generate a public-private key pair using Elliptic Curve Cryptography (ECC). Use the ecdsa library to generate the keys.

- create_transaction(receiver, amount): Create a new transaction, sign it with the wallet's private key using the sign_transaction() method, and return the signed transaction.

- verify_signature(transaction): Verify the signature of a given transaction using the wallet's public key and the verify_transaction() method.

In [36]:
class Wallet:

    # Initialize wallet with empty keys
    def __init__(self):
        self.private_key = None
        self.public_key = None

    # Generate a public-private key pair using ECDSA
    def generate_keys(self):
        self.private_key = SigningKey.generate(curve=SECP256k1)
        self.public_key = self.private_key.get_verifying_key()

    # Create and sign a transaction using the private key
    def create_transaction(self, receiver, amount):
        tx = Transaction(str(self.public_key), receiver, amount)
        tx.sign_transaction(self.private_key)
        return tx

    # Verify the signature of a transaction using the public key
    def verify_signature(self, transaction):
        return transaction.verify_signature(self.public_key)

Question 11 : Validating Digital Signatures

- Generate a public-private key pair using the Wallet class.

- Create a sample Transaction object, sign it using the private key, and then verify the signature using the public key.

- Modify the amount field of the Transaction object and attempt to verify the signature again. Verify that the verify_signature() method correctly rejects the modified transaction.

In [37]:
wallet = Wallet()
wallet.generate_keys()

# Create transaction
tx = wallet.create_transaction("Bob", 100)

print("Valid signature:", wallet.verify_signature(tx))

# Modify transaction
tx.amount = 200

print("After modification:", wallet.verify_signature(tx))

Valid signature: True
After modification: False


### Part 3 :  Blockchain Consensus



QUESTION 12 : Block Modification

Modify the Block class to include a difficulty attribute and a nonce field.

In [38]:
class Block:
    def __init__(self, block_number, timestamp, previous_hash, transactions, difficulty=2):
        # Block basic attributes
        self.block_number = block_number
        self.timestamp = timestamp
        self.previous_hash = previous_hash
        self.transactions = transactions

        # Mining parameters
        self.nonce = 0
        self.difficulty = difficulty

        # Initial hash calculation
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        import hashlib, json

        # Prepare block data for hashing
        data = {
            "block_number": self.block_number,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "transactions": self.transactions,
            "nonce": self.nonce
        }

        # Convert data to string and apply SHA-256
        block_string = json.dumps(data, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

QUESTION 13 : Mining

Implement a method in the Blockchain class called mine_block(transactions)
that takes a list of transactions as input, creates a new block, and then mines the block
by finding a valid nonce that produces a hash that corresponds to the difficulty target
(number of 0 at the beginning of the hash).

Use a while loop to try different nonce values until a valid hash is found.

In [39]:
class Blockchain:
    def __init__(self):
        # Initialize blockchain with genesis block
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        # First block in the chain
        return Block(0, 0, "0"*64, ["Genesis"], difficulty=2)

    def get_latest_block(self):
        # Return the last block
        return self.chain[-1]

    def add_block(self, block):
        # Link new block to previous block
        latest = self.get_latest_block()
        block.previous_hash = latest.hash
        block.hash = block.calculate_hash()
        self.chain.append(block)

    def mine_block(self, transactions):
        import time

        # Get last block
        latest = self.get_latest_block()

        # Create new block to mine
        new_block = Block(
            block_number=latest.block_number + 1,
            timestamp=int(time.time()),
            previous_hash=latest.hash,
            transactions=transactions,
            difficulty=2
        )

        # Define target based on difficulty
        target = "0" * new_block.difficulty

        # Mining loop: find valid nonce
        while not new_block.hash.startswith(target):
            new_block.nonce += 1
            new_block.hash = new_block.calculate_hash()

        # Add mined block to chain
        self.chain.append(new_block)

        return new_block

QUESTION 14 :  Mining Test

Test the mine_block(transactions) method and give the needed time to mine your block.

In [40]:
import time

# Create blockchain
bc = Blockchain()

# Start timing
start = time.time()

# Mine a block with sample transactions
mined_block = bc.mine_block(["Alice -> Bob", "Bob -> Charlie"])

# End timing
end = time.time()

# Display results
print("Block mined!")
print("Block number:", mined_block.block_number)
print("Hash:", mined_block.hash)
print("Nonce:", mined_block.nonce)
print("Time needed:", round(end - start, 4), "seconds")

Block mined!
Block number: 1
Hash: 00f99416c1b92086151cdba701b62d6e9254ed12774774ad41a9d5f28f6efcf9
Nonce: 228
Time needed: 0.002 seconds
